In [ ]:
import math
import os
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from tqdm import tqdm
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [ ]:
# Init Face Landmarker
base_options = python.BaseOptions(model_asset_path='model/face_landmarker_v2_with_blendshapes.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=True,
    num_faces=1
)
face_landmarker = vision.FaceLandmarker.create_from_options(options)
mp_holistic = mp.solutions.holistic

In [ ]:
#Extracts x, y, z coordinates from landmarks
def extract_landmarks(landmarks, prefix, num_landmarks):
    if landmarks:
        return {f"{prefix}_{axis}_{i+1}": getattr(landmark, axis)
                for i, landmark in enumerate(landmarks.landmark[:num_landmarks]) for axis in ['x', 'y', 'z']}
    else:
        return {f"{prefix}_{axis}_{i+1}": np.nan for i in range(num_landmarks) for axis in ['x', 'y', 'z']}

In [ ]:
def _clamp(value, min_value, max_value):
    return max(min_value, min(value, max_value))

In [ ]:
# Constants for face cropping
FACE_MIN_WIDTH = 50
FACE_MIN_HEIGHT = 50

def crop_face_region(frame_in: np.ndarray, body_landmarks: dict) -> dict:
    r_shoulder_x, r_shoulder_y, _ = body_landmarks.get("right_shoulder", [0, 0, 0])
    l_shoulder_x, l_shoulder_y, _ = body_landmarks.get("left_shoulder", [0, 0, 0])
    nose_tip_x, nose_tip_y, _ = body_landmarks.get("nose", [0, 0, 0])

    s_x = (r_shoulder_x + l_shoulder_x) / 2
    s_y = (r_shoulder_y + l_shoulder_y) / 2
    sn = math.sqrt((nose_tip_x - s_x) ** 2 + (nose_tip_y - s_y) ** 2)
    sn = int(sn * 0.75)

    x1 = nose_tip_x - sn
    x2 = nose_tip_x + sn
    y1 = nose_tip_y - sn
    y2 = nose_tip_y + sn

    h, w, _ = frame_in.shape
    x1 = _clamp(x1, 0, w - 1)
    x2 = _clamp(x2, 0, w - 1)
    y1 = _clamp(y1, 0, h - 1)
    y2 = _clamp(y2, 0, h - 1)

    cropped_frame = frame_in[y1:y2, x1:x2, :].copy()
    # bounds = (x1, y1, x2, y2)

    return cropped_frame

In [ ]:
# Find 'Front.mp4' files in subfolders.
def find_video(root_folder):
    video_files = []
    for root, _, files in os.walk(root_folder):
        for file in files:
            if file == 'Front.mp4':
                video_files.append(os.path.join(root, file))
    return video_files

In [ ]:
def process_video(video):
    # Init holistic model
    holistic_model = mp_holistic.Holistic(static_image_mode=False, model_complexity=1)
    
    cap = cv2.VideoCapture(video)
    data = []
    face_blendshapes_names = None
    
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    duration_sec = frame_count / fps if fps > 0 else np.nan
    
    video_metadata = {
        "video_name": os.path.basename(video),
        "frame_count": frame_count,
        "fps": fps,
        "duration_sec": duration_sec
    }
    
    for frame_num in range(frame_count):
        ret, frame = cap.read()
        if not ret:
            # Handle dropped frame
            frame_data = {'frame': frame_num + 1}
            frame_data.update({name: np.nan for name in frame_data})
            data.append(frame_data)
            continue

        # Convert to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_data = {'frame': frame_num + 1}

        # Holistic inference
        results = holistic_model.process(image)

        # Handle missing pose landmarks
        if results.pose_landmarks is None:
            frame_data.update({name: np.nan for name in frame_data})
            data.append(frame_data)
            continue

        # Extract left and right hand landmarks
        if results.left_hand_landmarks:
            for i, landmark in enumerate(results.left_hand_landmarks.landmark):
                hand_name = mp_holistic.HandLandmark(i).name
                frame_data[f"left_hand_{hand_name}_x"] = landmark.x
                frame_data[f"left_hand_{hand_name}_y"] = landmark.y
                frame_data[f"left_hand_{hand_name}_z"] = landmark.z
        
        if results.right_hand_landmarks:
            for i, landmark in enumerate(results.right_hand_landmarks.landmark):
                hand_name = mp_holistic.HandLandmark(i).name
                frame_data[f"right_hand_{hand_name}_x"] = landmark.x
                frame_data[f"right_hand_{hand_name}_y"] = landmark.y
                frame_data[f"right_hand_{hand_name}_z"] = landmark.z

        # Extract pose landmarks
        pose_landmarks = results.pose_landmarks
        for i in range(25):
            try:
                landmark = pose_landmarks.landmark[i]
                pose_name = mp_holistic.PoseLandmark(i).name
                frame_data[f"pose_{pose_name}_x"] = landmark.x
                frame_data[f"pose_{pose_name}_y"] = landmark.y
                frame_data[f"pose_{pose_name}_z"] = landmark.z
            except IndexError:
                frame_data[f"pose_{i}"] = np.nan

        # Extract body landmarks for cropping and face detection
        body_landmarks = {
            "right_shoulder": pose_landmarks.landmark[mp_holistic.PoseLandmark.RIGHT_SHOULDER],
            "left_shoulder": pose_landmarks.landmark[mp_holistic.PoseLandmark.LEFT_SHOULDER],
            "nose": pose_landmarks.landmark[mp_holistic.PoseLandmark.NOSE],
        }

        # Convert normalized to pixel
        h, w, _ = frame.shape
        for key, lm in body_landmarks.items():
            body_landmarks[key] = (int(lm.x * w), int(lm.y * h), int(lm.z * w))

        # Face cropping
        face_crop = crop_face_region(frame, body_landmarks)

        # Validate face region size
        face_h, face_w, _ = face_crop.shape

        # Face Blendshapes Processing
        if face_w >= FACE_MIN_WIDTH and face_h >= FACE_MIN_HEIGHT:
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB))
            face_results = face_landmarker.detect(mp_image)
            if face_results.face_blendshapes:
                # Extract and flatten transformation matrix
                if face_results.facial_transformation_matrixes:
                    transform_matrix = np.array(face_results.facial_transformation_matrixes[0]).reshape(4, 4)
                    flat_transform = transform_matrix.flatten()
                    for i, val in enumerate(flat_transform):
                        frame_data[f"transform_{i}"] = val
                else:
                    for i in range(16):
                        frame_data[f"transform_{i}"] = np.nan
                        
                if face_blendshapes_names is None:
                    face_blendshapes_names = [blendshape.category_name for blendshape in face_results.face_blendshapes[0]]
                
                face_blendshapes_scores = [blendshape.score for blendshape in face_results.face_blendshapes[0]]
                frame_data.update({name: score for name, score in zip(face_blendshapes_names, face_blendshapes_scores)})
            else:
                # Fill NaNs if blendshapes are missing
                if face_blendshapes_names:
                    frame_data.update({name: np.nan for name in face_blendshapes_names})
        
        data.append(frame_data)
        
    cap.release()
    holistic_model.close()

    return pd.DataFrame(data), video_metadata

In [ ]:
def extract_face_transform(result):
    if result.facial_transformation_matrixes:
        mat = np.array(result.facial_transformation_matrixes[0]).reshape(4, 4)
        return mat
    else:
        return np.full((4, 4), np.nan)

In [ ]:
def interpolate_missing_values(csv_path):
    df = pd.read_csv(csv_path)
    df = df.interpolate(method='linear', axis=0, limit_direction='both')
    
    return df

In [ ]:
def compute_derivatives(df, landmarks, axes=['x', 'y', 'z']):
    new_cols = {}

    for lm in landmarks:
        for axis in axes:
            col = f"{lm}_{axis}"
            if col in df.columns:
                s = df[col].diff().fillna(0)
                new_cols[f"{col}_velocity"] = s
                new_cols[f"{col}_acceleration"] = s.diff().fillna(0)

    if new_cols:
        df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)

    return df

In [ ]:
# Mean and std deviation function
def compute_stats(df, landmarks, blendshapes, axes=['x', 'y', 'z']):
    stats = {}

    # Pose & Hand landmarks
    for lm in landmarks:
        for axis in axes:
            col_name = f"{lm}_{axis}"
            vel_col = f"{col_name}_velocity"
            acc_col = f"{col_name}_acceleration"

            if col_name in df.columns:
                stats[f"{col_name}_mean"] = df[col_name].mean()
                stats[f"{col_name}_std"] = df[col_name].std()
            if vel_col in df.columns:
                stats[f"{vel_col}_mean"] = df[vel_col].mean()
                stats[f"{vel_col}_std"] = df[vel_col].std()
            if acc_col in df.columns:
                stats[f"{acc_col}_mean"] = df[acc_col].mean()
                stats[f"{acc_col}_std"] = df[acc_col].std()

    # Facial Blendshapes
    for blendshape in blendshapes:
        if blendshape in df.columns:
            stats[f"{blendshape}_mean"] = df[blendshape].mean()
            stats[f"{blendshape}_std"] = df[blendshape].std()

    return pd.DataFrame([stats])

In [ ]:
def compute_additional_features(csv_path, video_name, output_folder, meta):
    df = pd.read_csv(csv_path)
    duration_sec = meta.get("duration_sec", np.nan) if meta else np.nan

    # Landmark and blendshape sets
    pose_hand_landmarks = [
        "left_hand_WRIST", "right_hand_WRIST",
        "pose_LEFT_ELBOW", "pose_RIGHT_ELBOW",
        "pose_LEFT_HIP", "pose_RIGHT_HIP",
        "pose_LEFT_SHOULDER", "pose_RIGHT_SHOULDER",
        "pose_NOSE",
    ]

    facial_blendshapes = [
        "_neutral", "browDownLeft", "browDownRight", "browInnerUp",
        "browOuterUpLeft", "browOuterUpRight", "cheekPuff",
        "cheekSquintLeft", "cheekSquintRight", "eyeBlinkLeft",
        "eyeBlinkRight", "eyeLookDownLeft", "eyeLookDownRight",
        "eyeLookInLeft", "eyeLookInRight", "eyeLookOutLeft",
        "eyeLookOutRight", "eyeLookUpLeft", "eyeLookUpRight",
        "eyeSquintLeft", "eyeSquintRight", "eyeWideLeft", "eyeWideRight",
        "jawForward", "jawLeft", "jawOpen", "jawRight", "mouthClose",
        "mouthDimpleLeft", "mouthDimpleRight", "mouthFrownLeft",
        "mouthFrownRight", "mouthFunnel", "mouthLeft",
        "mouthLowerDownLeft", "mouthLowerDownRight", "mouthPressLeft",
        "mouthPressRight", "mouthPucker", "mouthRight",
        "mouthRollLower", "mouthRollUpper", "mouthShrugLower",
        "mouthShrugUpper", "mouthSmileLeft", "mouthSmileRight",
        "mouthStretchLeft", "mouthStretchRight", "mouthUpperUpLeft",
        "mouthUpperUpRight", "noseSneerLeft", "noseSneerRight"
    ]

    # Motion (velocity/acceleration)
    df = compute_derivatives(df, pose_hand_landmarks)
    stats_df = compute_stats(df, pose_hand_landmarks, facial_blendshapes)

    rotation_cols = [c for c in df.columns if c in [
        "head_pitch_deg", "head_yaw_deg", "head_roll_deg",
        "left_arm_angle", "right_arm_angle", "torso_pitch", "torso_roll", "torso_yaw"
    ]]

    computed_pairs = [
        "dist_wrist_lr",
        "dist_left_wrist_to_left_shoulder",
        "dist_right_wrist_to_right_shoulder",
        "dist_elbows_lr",
        "dist_left_wrist_to_nose",
        "dist_right_wrist_to_nose",
        "dist_left_wrist_to_right_shoulder",
        "dist_right_wrist_to_left_shoulder"
    ]
    distance_cols = [c for c in computed_pairs if c in df.columns]
    motion_cols   = [c for c in df.columns if c.endswith("_accum_dist")]

    # Summarization
    def summarize_meanstd(cols):
        out = {}
        for col in cols:
            s = pd.to_numeric(df[col], errors="coerce")
            out[f"{col}__mean"] = s.mean()
            out[f"{col}__std"]  = s.std()
        return out

    def summarize_final(accum_cols):
        out = {}
        for col in accum_cols:
            s = pd.to_numeric(df[col], errors="coerce")
            out[f"{col}__avg"] = s.mean()   # use average of accumulated distance
        return out

    # Combine all features
    ext = {}
    ext.update(summarize_meanstd(rotation_cols))
    ext.update(summarize_final(distance_cols))
    ext.update(summarize_final(motion_cols))

    combined = pd.concat([stats_df, pd.DataFrame([ext])], axis=1)

    # Save
    updated_csv_path = os.path.join(output_folder, f"{video_name}_mediapipe_data_interpolated.csv")
    df.to_csv(updated_csv_path, index=False)

    stats_csv_path = os.path.join(output_folder, f"{video_name}_features.csv")
    combined.to_csv(stats_csv_path, index=False)

    print(f"Processed CSV saved to: {updated_csv_path}")
    print(f"Flattened features for ML saved to: {stats_csv_path}")


In [ ]:
def compute_transform_features(csv_path):
    df = pd.read_csv(csv_path)

    # Check if facial transform exists
    transform_cols = [f"transform_{i}" for i in range(16)]
    has_face_transform = all(col in df.columns for col in transform_cols)

    def rotation_matrix_to_euler_angles(R):
        sy = np.sqrt(R[0, 0]**2 + R[1, 0]**2)
        singular = sy < 1e-6
        if not singular:
            x = np.arctan2(R[2, 1], R[2, 2])
            y = np.arctan2(-R[2, 0], sy)
            z = np.arctan2(R[1, 0], R[0, 0])
        else:
            x = np.arctan2(-R[1, 2], R[1, 1])
            y = np.arctan2(-R[2, 0], sy)
            z = 0
        return np.degrees([x, y, z])

    # Compute 3D angle between two vectors
    def angle_between(v1, v2):
        v1, v2 = np.array(v1), np.array(v2)
        dot = np.dot(v1, v2)
        norm = np.linalg.norm(v1) * np.linalg.norm(v2)
        if norm == 0: 
            return np.nan
        return np.degrees(np.arccos(np.clip(dot / norm, -1.0, 1.0)))

    head_pitch, head_yaw, head_roll = [], [], []
    left_arm_angle, right_arm_angle = [], []
    torso_pitch, torso_roll, torso_yaw = [], [], []

    prev_t = None

    for i, row in df.iterrows():
        # Head rotation
        if has_face_transform:
            try:
                mat = row[transform_cols].to_numpy().reshape(4, 4)
                R = mat[:3, :3]
                t = mat[:3, 3]
                pitch, yaw, roll = rotation_matrix_to_euler_angles(R)

                prev_t = t
            except Exception:
                pitch, yaw, roll = np.nan, np.nan, np.nan
        else:
            pitch, yaw, roll = np.nan, np.nan, np.nan

        head_pitch.append(pitch)
        head_yaw.append(yaw)
        head_roll.append(roll)

        # Arm rotation
        try:
            # left arm: shoulder -> elbow
            l_shoulder = np.array([row["pose_LEFT_SHOULDER_x"], row["pose_LEFT_SHOULDER_y"], row["pose_LEFT_SHOULDER_z"]])
            l_elbow    = np.array([row["pose_LEFT_ELBOW_x"], row["pose_LEFT_ELBOW_y"], row["pose_LEFT_ELBOW_z"]])
            left_vec = l_elbow - l_shoulder

            # right arm: shoulder -> elbow
            r_shoulder = np.array([row["pose_RIGHT_SHOULDER_x"], row["pose_RIGHT_SHOULDER_y"], row["pose_RIGHT_SHOULDER_z"]])
            r_elbow    = np.array([row["pose_RIGHT_ELBOW_x"], row["pose_RIGHT_ELBOW_y"], row["pose_RIGHT_ELBOW_z"]])
            right_vec = r_elbow - r_shoulder

            # Torso plane: shoulder to hip vectors
            l_hip = np.array([row["pose_LEFT_HIP_x"], row["pose_LEFT_HIP_y"], row["pose_LEFT_HIP_z"]])
            r_hip = np.array([row["pose_RIGHT_HIP_x"], row["pose_RIGHT_HIP_y"], row["pose_RIGHT_HIP_z"]])
            torso_vec_vert = ((l_hip + r_hip) / 2) - ((l_shoulder + r_shoulder) / 2)
            torso_vec_horiz = r_shoulder - l_shoulder

            # Compute rotation
            left_angle = angle_between(left_vec, torso_vec_vert)
            right_angle = angle_between(right_vec, torso_vec_vert)
            torso_pitch_angle = angle_between(torso_vec_vert, np.array([0, -1, 0]))  # deviation from vertical
            torso_roll_angle = angle_between(torso_vec_horiz, np.array([1, 0, 0]))   # shoulder tilt
        
            dx = r_shoulder[0] - l_shoulder[0]
            dz = r_shoulder[2] - l_shoulder[2]
            torso_yaw_angle = np.degrees(np.arctan2(dz, dx))
        except Exception:
            left_angle = right_angle = torso_pitch_angle = torso_roll_angle = torso_yaw_angle = np.nan


        left_arm_angle.append(left_angle)
        right_arm_angle.append(right_angle)
        torso_pitch.append(torso_pitch_angle)
        torso_roll.append(torso_roll_angle)
        torso_yaw.append(torso_yaw_angle)


    df["head_pitch_deg"] = head_pitch
    df["head_yaw_deg"] = head_yaw
    df["head_roll_deg"] = head_roll
    df["left_arm_angle"] = left_arm_angle
    df["right_arm_angle"] = right_arm_angle
    df["torso_pitch"] = torso_pitch
    df["torso_roll"] = torso_roll
    df["torso_yaw"] = torso_yaw

    # Save
    df.to_csv(csv_path, index=False)
    print(f"Interpolated file with rotations: {csv_path}")
    return df


In [ ]:
def augment_distance(csv_path):
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"[WARN] Failed to read {csv_path}: {e}")
        return None

    COLS = {
        "L_WRIST": ("left_hand_WRIST_x", "left_hand_WRIST_y", "left_hand_WRIST_z"),
        "R_WRIST": ("right_hand_WRIST_x", "right_hand_WRIST_y", "right_hand_WRIST_z"),
        "L_ELBOW": ("pose_LEFT_ELBOW_x", "pose_LEFT_ELBOW_y", "pose_LEFT_ELBOW_z"),
        "R_ELBOW": ("pose_RIGHT_ELBOW_x", "pose_RIGHT_ELBOW_y", "pose_RIGHT_ELBOW_z"),
        "L_SHOULDER": ("pose_LEFT_SHOULDER_x", "pose_LEFT_SHOULDER_y", "pose_LEFT_SHOULDER_z"),
        "R_SHOULDER": ("pose_RIGHT_SHOULDER_x", "pose_RIGHT_SHOULDER_y", "pose_RIGHT_SHOULDER_z"),
        "NOSE": ("pose_NOSE_x", "pose_NOSE_y", "pose_NOSE_z"),
    }

    DIST_COLS = {
        "dist_wrist_lr": ("L_WRIST", "R_WRIST"),
        "dist_left_wrist_to_left_shoulder": ("L_WRIST", "L_SHOULDER"),
        "dist_right_wrist_to_right_shoulder": ("R_WRIST", "R_SHOULDER"),
        "dist_left_wrist_to_nose": ("L_WRIST", "NOSE"),
        "dist_right_wrist_to_nose": ("R_WRIST", "NOSE"),
        "dist_left_wrist_to_right_shoulder": ("L_WRIST", "R_SHOULDER"),
        "dist_right_wrist_to_left_shoulder": ("R_WRIST", "L_SHOULDER"),
        "dist_elbows_lr": ("L_ELBOW", "R_ELBOW"),
    }

    def dist3d(df, p1_key, p2_key):
        x1, y1, z1 = COLS[p1_key]
        x2, y2, z2 = COLS[p2_key]
        if not all(c in df.columns for c in (x1, y1, z1, x2, y2, z2)):
            return pd.Series(np.nan, index=df.index)
        a = df[[x1, y1, z1]].to_numpy(dtype=float)
        b = df[[x2, y2, z2]].to_numpy(dtype=float)
        return np.linalg.norm(a - b, axis=1)

    def dist3d_pointwise(df, point_key):
        x, y, z = COLS[point_key]
        if not all(c in df.columns for c in (x, y, z)):
            return pd.Series(np.nan, index=df.index)
        p = df[[x, y, z]].to_numpy(dtype=float)
        diff = np.linalg.norm(np.diff(p, axis=0), axis=1)
        return pd.Series(np.concatenate([[0.0], diff]), index=df.index)

    # Pairwise distances
    for out_col, (p1, p2) in DIST_COLS.items():
        df[out_col] = dist3d(df, p1, p2)

    # Landmark motion (frame-to-frame + accumulated)
    for point_key in ["L_WRIST", "R_WRIST", "L_SHOULDER", "R_SHOULDER", "NOSE"]:
        acc_col = f"{point_key}_accum_dist"

        step = dist3d_pointwise(df, point_key)
        accum = step.cumsum()

        df[acc_col] = accum

  
    df.to_csv(csv_path, index=False)
    print(f"Added distance: {os.path.basename(csv_path)}")
    return df


In [ ]:
def save_processed_data(df, root_folder, video_path, meta=None):
    output_folder = os.path.join(root_folder, "Output", os.path.dirname(os.path.relpath(video_path, root_folder)))
    os.makedirs(output_folder, exist_ok=True)

    video_name = os.path.splitext(os.path.basename(video_path))[0]
    output_csv = os.path.join(output_folder, f"{video_name}_mediapipe_data.csv")
    df.to_csv(output_csv, index=False)
    print(f"Saved raw features: {output_csv}")

    df_interpolation = interpolate_missing_values(output_csv)
    interpolated_csv = os.path.join(output_folder, f"{video_name}_mediapipe_data_interpolated.csv")
    df_interpolation.to_csv(interpolated_csv, index=False)
    print(f"Saved interpolated data: {interpolated_csv}")

    if os.path.exists(interpolated_csv):
        print("Adding rotation & movement features to interpolated data...")
        compute_transform_features(interpolated_csv)

        print("Adding distance and joint motion features...")
        augment_distance(interpolated_csv)

        print("Proceeding to compute additional features...")
        compute_additional_features(interpolated_csv, video_name, output_folder, meta)


In [ ]:
root_folder = "/project/" #change folder

In [ ]:
video_files = find_video(root_folder)

for video_file in tqdm(video_files, desc="Processing Videos"):
    df, meta = process_video(video_file)
    save_processed_data(df, root_folder, video_file, meta)


In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import find_peaks


BLENDSHAPES = [
    "_neutral", "browDownLeft", "browDownRight", "browInnerUp",
    "browOuterUpLeft", "browOuterUpRight", "cheekPuff",
    "cheekSquintLeft", "cheekSquintRight", "eyeBlinkLeft",
    "eyeBlinkRight", "eyeLookDownLeft", "eyeLookDownRight",
    "eyeLookInLeft", "eyeLookInRight", "eyeLookOutLeft",
    "eyeLookOutRight", "eyeLookUpLeft", "eyeLookUpRight",
    "eyeSquintLeft", "eyeSquintRight", "eyeWideLeft", "eyeWideRight",
    "jawForward", "jawLeft", "jawOpen", "jawRight", "mouthClose",
    "mouthDimpleLeft", "mouthDimpleRight", "mouthFrownLeft",
    "mouthFrownRight", "mouthFunnel", "mouthLeft",
    "mouthLowerDownLeft", "mouthLowerDownRight", "mouthPressLeft",
    "mouthPressRight", "mouthPucker", "mouthRight",
    "mouthRollLower", "mouthRollUpper", "mouthShrugLower",
    "mouthShrugUpper", "mouthSmileLeft", "mouthSmileRight",
    "mouthStretchLeft", "mouthStretchRight", "mouthUpperUpLeft",
    "mouthUpperUpRight", "noseSneerLeft", "noseSneerRight"
]

ANGLES = [
    "head_pitch_deg", "head_yaw_deg", "head_roll_deg",
    "left_arm_angle", "right_arm_angle",
    "torso_pitch", "torso_roll", "torso_yaw",
]

DISTANCES = [
    "dist_wrist_lr", "dist_elbows_lr",
    "dist_left_wrist_to_left_shoulder", "dist_right_wrist_to_right_shoulder",
    "dist_left_wrist_to_nose", "dist_right_wrist_to_nose",
    "dist_left_wrist_to_right_shoulder", "dist_right_wrist_to_left_shoulder",
]

POSE = [
    "pose_NOSE",
    "pose_LEFT_SHOULDER", "pose_RIGHT_SHOULDER",
    "pose_LEFT_ELBOW", "pose_RIGHT_ELBOW",
    "pose_LEFT_WRIST", "pose_RIGHT_WRIST",
    "pose_LEFT_HIP", "pose_RIGHT_HIP",
]

HAND_WRISTS = ["left_hand_WRIST", "right_hand_WRIST"]


def select_peak_features(df: pd.DataFrame) -> list[str]:
    cols: list[str] = []

    cols.extend([c for c in BLENDSHAPES if c in df.columns])

    for base in POSE:
        for axis in ("_x", "_y", "_z"):
            c = f"{base}{axis}"
            if c in df.columns:
                cols.append(c)

    for base in HAND_WRISTS:
        for axis in ("_x", "_y", "_z"):
            c = f"{base}{axis}"
            if c in df.columns:
                cols.append(c)

    cols.extend([c for c in ANGLES if c in df.columns])
    cols.extend([c for c in DISTANCES if c in df.columns])

    def is_excluded(name: str) -> bool:
        return (
            name.startswith("transform_")
            or name.endswith("_accum_dist")
            or name.endswith("_velocity")
            or name.endswith("_acceleration")
        )

    cols = [c for c in cols if not is_excluded(c)]

    seen: set[str] = set()
    out: list[str] = []
    for c in cols:
        if c not in seen:
            out.append(c)
            seen.add(c)
    return out


def add_peak_columns_inplace(
    csv_path: str | Path,
    *,
    prominence: float | None = None,
    overwrite: bool = True,
) -> None:
    csv_path = Path(csv_path)
    df = pd.read_csv(csv_path)

    peak_features = select_peak_features(df)

    for feat in peak_features:
        x = pd.to_numeric(df[feat], errors="coerce").to_numpy(dtype=float)

        if np.isnan(x).any():
            continue

        peaks, _ = find_peaks(x, prominence=prominence)
        ind = np.zeros(len(df), dtype=int)
        ind[peaks] = 1
        df[f"{feat}__peak"] = ind


    out_path = csv_path if overwrite else csv_path.with_name(csv_path.stem + "_with_peaks.csv")
    df.to_csv(out_path, index=False)


def add_peaks_to_all_interpolated_csvs(
    output_root: str | Path = "Output",
    *,
    prominence: float | None = None,
) -> None:
    output_root = Path(output_root)
    paths = sorted(output_root.rglob("*_mediapipe_data_interpolated.csv"))

    for p in paths:
        add_peak_columns_inplace(p, prominence=prominence, overwrite=True)
        print(f"peaks added: {p}")


add_peaks_to_all_interpolated_csvs("/project/Output", prominence=None)

In [ ]:
import os
import numpy as np
import pandas as pd

def flatten_from_interpolated_csv(
    csv_path: str,
    video_name: str,
    output_folder: str,
    meta: dict | None,
):

    df = pd.read_csv(csv_path)

    duration_sec = meta.get("duration_sec", np.nan) if meta else np.nan

    pose_hand_landmarks = [
        "left_hand_WRIST", "right_hand_WRIST",
        "pose_LEFT_ELBOW", "pose_RIGHT_ELBOW",
        "pose_LEFT_HIP", "pose_RIGHT_HIP",
        "pose_LEFT_SHOULDER", "pose_RIGHT_SHOULDER",
        "pose_NOSE",
    ]

    facial_blendshapes = [
        "_neutral", "browDownLeft", "browDownRight", "browInnerUp",
        "browOuterUpLeft", "browOuterUpRight", "cheekPuff",
        "cheekSquintLeft", "cheekSquintRight", "eyeBlinkLeft",
        "eyeBlinkRight", "eyeLookDownLeft", "eyeLookDownRight",
        "eyeLookInLeft", "eyeLookInRight", "eyeLookOutLeft",
        "eyeLookOutRight", "eyeLookUpLeft", "eyeLookUpRight",
        "eyeSquintLeft", "eyeSquintRight", "eyeWideLeft", "eyeWideRight",
        "jawForward", "jawLeft", "jawOpen", "jawRight", "mouthClose",
        "mouthDimpleLeft", "mouthDimpleRight", "mouthFrownLeft",
        "mouthFrownRight", "mouthFunnel", "mouthLeft",
        "mouthLowerDownLeft", "mouthLowerDownRight", "mouthPressLeft",
        "mouthPressRight", "mouthPucker", "mouthRight",
        "mouthRollLower", "mouthRollUpper", "mouthShrugLower",
        "mouthShrugUpper", "mouthSmileLeft", "mouthSmileRight",
        "mouthStretchLeft", "mouthStretchRight", "mouthUpperUpLeft",
        "mouthUpperUpRight", "noseSneerLeft", "noseSneerRight"
    ]

    rotation_cols = [c for c in [
        "head_pitch_deg", "head_yaw_deg", "head_roll_deg",
        "left_arm_angle", "right_arm_angle",
        "torso_pitch", "torso_roll", "torso_yaw"
    ] if c in df.columns]

    computed_pairs = [
        "dist_wrist_lr",
        "dist_left_wrist_to_left_shoulder",
        "dist_right_wrist_to_right_shoulder",
        "dist_elbows_lr",
        "dist_left_wrist_to_nose",
        "dist_right_wrist_to_nose",
        "dist_left_wrist_to_right_shoulder",
        "dist_right_wrist_to_left_shoulder"
    ]
    distance_cols = [c for c in computed_pairs if c in df.columns]
    motion_cols = [c for c in df.columns if c.endswith("_accum_dist")]

    def summarize_meanstd(cols):
        out = {}
        for col in cols:
            s = pd.to_numeric(df[col], errors="coerce")
            out[f"{col}_mean"] = s.mean()
            out[f"{col}_std"]  = s.std()
        return out

    def summarize_avg(cols):
        out = {}
        for col in cols:
            s = pd.to_numeric(df[col], errors="coerce")
            out[f"{col}_avg"] = s.mean()
        return out

    stats_df = compute_stats(df, pose_hand_landmarks, facial_blendshapes)

    ext = {}
    ext.update(summarize_meanstd(rotation_cols))
    ext.update(summarize_avg(distance_cols))
    ext.update(summarize_avg(motion_cols))

    peak_cols = [c for c in df.columns if c.endswith("__peak")]
    peak_ext = {}
    for pc in peak_cols:
        s = pd.to_numeric(df[pc], errors="coerce").fillna(0)
        count = float(s.sum())
        # peak_ext[pc.replace("__peak", "_peaks_count")] = count
        peak_ext[pc.replace("__peak", "_peaks_per_s")] = (
            count / duration_sec if duration_sec and duration_sec > 0 else np.nan
        )
    ext.update(peak_ext)

    combined = pd.concat([stats_df, pd.DataFrame([ext])], axis=1)

    stats_csv_path = os.path.join(output_folder, f"{video_name}_features.csv")
    combined.to_csv(stats_csv_path, index=False)

    print(f"Flattened features for ML saved to: {stats_csv_path}")
    

In [ ]:
def save_flatten_features_from_interpolated(root_folder):
    FPS = 30.0
    root = Path(root_folder)
    paths = sorted(root.rglob("*_mediapipe_data_interpolated.csv"))

    for csv_path in paths:

        csv_path = Path(csv_path)
        df = pd.read_csv(csv_path)

        video_name = csv_path.name.replace("_mediapipe_data_interpolated.csv", "")
        output_folder = csv_path.parent

        # compute duration from frame count
        n_frames = len(df)
        duration_sec = n_frames / FPS if FPS > 0 else None

        meta = {"duration_sec": duration_sec}

        flatten_from_interpolated_csv(
            csv_path=str(csv_path),
            video_name=video_name,
            output_folder=str(output_folder),
            meta=meta
        )

        print(f"Flatten features created: {video_name}  duration={duration_sec:.2f}s")

save_flatten_features_from_interpolated("/project/Output")